# LLM judge pipeline for `results.jsonl`

Minimal Colab notebook version of the cleaned script.

This notebook:
1. Loads `results.jsonl`
2. Builds OpenAI Batch requests for LLM judging
3. Submits the batch
4. Downloads the output
5. Rewrites `baseline_substring_match`, `xrag_substring_match`, and `overflow`
6. Drops partial / ambiguous (`0.5`) cases
7. Saves:
   - `results_llm_scored.jsonl`
   - `results_llm_filtered.jsonl`


In [1]:
!pip -q install openai pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:

import json
import os
import time
from pathlib import Path
from typing import Any

import pandas as pd
from openai import OpenAI

JUDGE_MODEL = "gpt-5-mini"
RESULTS_PATH = "results.jsonl"
BATCH_JSONL = "judge_batch.jsonl"
JUDGE_OUT_JSONL = "judge_out.jsonl"
FULL_OUTPUT_JSONL = "results_llm_scored.jsonl"
FILTERED_OUTPUT_JSONL = "results_llm_filtered.jsonl"


## Set API key

In [ ]:

# Option 1:
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

# Option 2 in Colab:
from getpass import getpass
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


## Helper functions

In [ ]:

def get_client(api_key: str | None = None) -> OpenAI:
    key = api_key or os.environ.get("OPENAI_API_KEY")
    if not key:
        raise EnvironmentError("OPENAI_API_KEY is not set.")
    return OpenAI(api_key=key)


def load_results(path: str) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no} of {path}: {e}") from e

    if not rows:
        raise ValueError(f"No rows found in {path}")

    df = pd.DataFrame(rows)
    if "id" not in df.columns:
        df["id"] = [str(i) for i in range(len(df))]
    else:
        df["id"] = df["id"].astype(str)
    return df


def normalize_gold_answer(value: Any) -> str:
    if isinstance(value, list):
        if not value:
            return ""
        first = value[0]
        return "" if first is None else str(first)
    if value is None:
        return ""
    if pd.isna(value):
        return ""
    return str(value)


def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value)


def judge_system_prompt() -> str:
    return (
        "You are a strict grader scoring whether model predictions match a gold answer.\n\n"
        'Return ONLY a JSON object with exactly these keys:\n'
        '- "xrag": a number 0, 0.5, or 1\n'
        '- "baseline": a number 0, 0.5, or 1\n\n'
        'Use this rubric:\n'
        '- 1 (correct): prediction clearly and unambiguously contains the gold answer or an equivalent paraphrase.\n'
        '  Accept synonyms/paraphrases; for numbers allow equivalent forms (e.g., 0.2 = 20%) and minor rounding that does not change meaning.\n'
        '  If the gold answer is a list/set of required items, the prediction must contain all required elements for 1.\n'
        '- 0.5 (partial): some correct elements but incomplete/vague/missing required parts.\n'
        '- 0 (incorrect): empty/non-answer/irrelevant/refusal/no match.\n'
        '  If a prediction includes both a correct answer and a contradictory/conflicting answer, score 0.\n\n'
        'Rules:\n'
        '- Gold answer is the only source of truth.\n'
        '- Ignore casing/punctuation/formatting differences.\n'
        '- Do not output any text outside the JSON object.\n'
    )


def build_user_prompt(question: str, gold_answer: str, xrag_pred: str, baseline_pred: str) -> str:
    parts = []
    question = normalize_text(question).strip()
    if question:
        parts.append(f"QUESTION:\n{question}\n")

    parts.append(f"GOLD ANSWER:\n{gold_answer.strip()}\n")
    parts.append(f"XRAG PREDICTION:\n{xrag_pred.strip()}\n")
    parts.append(f"BASELINE PREDICTION:\n{baseline_pred.strip()}\n")
    parts.append(
        'Score each prediction vs the gold answer. Output only JSON:\n'
        '{"xrag": <0|0.5|1>, "baseline": <0|0.5|1>}'
    )
    return "\n".join(parts)


def build_batch_file(df: pd.DataFrame, batch_jsonl_path: str, judge_model: str = JUDGE_MODEL) -> None:
    required = {"answer", "baseline_pred", "xrag_pred"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns in results file: {sorted(missing)}")

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            gold_answer = normalize_gold_answer(row.get("answer"))
            xrag_pred = normalize_text(row.get("xrag_pred"))
            baseline_pred = normalize_text(row.get("baseline_pred"))
            question = normalize_text(row.get("question", ""))
            custom_id = f"judge::{row['id']}"

            job = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": judge_model,
                    "messages": [
                        {"role": "system", "content": judge_system_prompt()},
                        {
                            "role": "user",
                            "content": build_user_prompt(
                                question=question,
                                gold_answer=gold_answer,
                                xrag_pred=xrag_pred,
                                baseline_pred=baseline_pred,
                            ),
                        },
                    ],
                },
            }
            f.write(json.dumps(job, ensure_ascii=False) + "\n")


def submit_batch(batch_jsonl_path: str, api_key: str | None = None) -> tuple[str, str]:
    client = get_client(api_key)
    with open(batch_jsonl_path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    return uploaded.id, batch.id


def wait_for_batch(batch_id: str, api_key: str | None = None, poll_sec: int = 15):
    client = get_client(api_key)
    while True:
        batch = client.batches.retrieve(batch_id)
        print("batch status:", batch.status)
        if batch.status in {"completed", "failed", "cancelled", "expired"}:
            return batch
        time.sleep(poll_sec)


def download_batch_output(batch_id: str, out_path: str, api_key: str | None = None) -> str:
    client = get_client(api_key)
    batch = client.batches.retrieve(batch_id)
    if batch.status != "completed":
        raise RuntimeError(f"Batch is not completed. Current status: {batch.status}")
    if not batch.output_file_id:
        raise RuntimeError("Batch completed but output_file_id is missing")

    content = client.files.content(batch.output_file_id)
    with open(out_path, "wb") as f:
        f.write(content.read())
    return out_path


def safe_json_loads(text: str):
    try:
        return json.loads(text)
    except Exception:
        return None


def parse_batch_output(out_path: str) -> pd.DataFrame:
    rows = []
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            custom_id = obj.get("custom_id", "")
            response_body = (obj.get("response") or {}).get("body") or {}
            choices = response_body.get("choices") or []
            content = ""
            if choices:
                message = choices[0].get("message") or {}
                content = normalize_text(message.get("content", "")).strip()

            parsed = safe_json_loads(content)
            xrag_score = parsed.get("xrag") if parsed else None
            baseline_score = parsed.get("baseline") if parsed else None

            rows.append(
                {
                    "custom_id": custom_id,
                    "id": custom_id.replace("judge::", "", 1),
                    "raw_judge_content": content,
                    "parse_ok": parsed is not None,
                    "xrag_llm_score": xrag_score,
                    "baseline_llm_score": baseline_score,
                }
            )

    if not rows:
        raise ValueError(f"No judge rows found in {out_path}")
    return pd.DataFrame(rows)


def score_to_binary(score: Any):
    if score in (0, 0.0, 1, 1.0):
        return int(score)
    return None


def apply_judgments(results_path: str, judge_out_path: str):
    results_df = load_results(results_path)
    judge_df = parse_batch_output(judge_out_path)

    merged = results_df.merge(
        judge_df[["id", "parse_ok", "raw_judge_content", "xrag_llm_score", "baseline_llm_score"]],
        on="id",
        how="left",
        validate="1:1",
    )

    merged["baseline_substring_match_original"] = merged.get("baseline_substring_match")
    merged["xrag_substring_match_original"] = merged.get("xrag_substring_match")
    merged["overflow_original"] = merged.get("overflow")

    merged["baseline_llm_binary"] = merged["baseline_llm_score"].apply(score_to_binary)
    merged["xrag_llm_binary"] = merged["xrag_llm_score"].apply(score_to_binary)
    merged["judge_binary_ok"] = (
        merged["parse_ok"].fillna(False)
        & merged["baseline_llm_binary"].notna()
        & merged["xrag_llm_binary"].notna()
    )

    filtered = merged[merged["judge_binary_ok"]].copy()
    filtered["baseline_substring_match"] = filtered["baseline_llm_binary"].astype(int)
    filtered["xrag_substring_match"] = filtered["xrag_llm_binary"].astype(int)
    filtered["overflow_label"] = (
        (filtered["baseline_substring_match"] == 1)
        & (filtered["xrag_substring_match"] == 0)
    ).astype(int)

    return merged, filtered


def write_jsonl(df: pd.DataFrame, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)


## Load and inspect `results.jsonl`

In [ ]:

df = load_results(RESULTS_PATH)
print("rows:", len(df))
print("columns:", list(df.columns))
df.head()


In [ ]:
len(df)

## Build batch JSONL

In [ ]:

build_batch_file(df, BATCH_JSONL, judge_model=JUDGE_MODEL)
print("Saved:", BATCH_JSONL)


## Submit batch

In [ ]:

input_file_id, batch_id = submit_batch(BATCH_JSONL)
print("input_file_id:", input_file_id)
print("batch_id:", batch_id)


## Wait for completion

In [ ]:

# Paste your batch id here if needed
# batch_id = "batch_..."

final_batch = wait_for_batch(batch_id, poll_sec=15)
print("final status:", final_batch.status)


## Download batch output

In [ ]:

download_batch_output(batch_id, JUDGE_OUT_JSONL)
print("Saved:", JUDGE_OUT_JSONL)


## Apply judgments and drop partial (`0.5`) rows

In [ ]:
full_df, filtered_df = apply_judgments(RESULTS_PATH, JUDGE_OUT_JSONL)

print("full rows:", len(full_df))
print("filtered rows:", len(filtered_df))
print("dropped rows:", len(full_df) - len(filtered_df))

filtered_df.head()


In [11]:
filtered_df = filtered_df[filtered_df["baseline_substring_match"] == 1].copy()

print("After baseline==1 filter:", len(filtered_df))

After baseline==1 filter: 5706


### Concatenate all samples into single samples.jsonl

In [2]:
import json

input_files = [
#"/app/overflow-detection/scripts/data_preprocessing/runs/hotpotqa_moe/samples.jsonl",
"/app/overflow-detection/scripts/data_preprocessing/runs/squad_7b/samples.jsonl",
"/app/overflow-detection/scripts/data_preprocessing/runs/trivia_7b/samples.jsonl",
]

output_file = "/app/overflow-detection/scripts/data_preprocessing/runs/merged2_7b/samples.jsonl"

with open(output_file, "w", encoding="utf-8") as fout:
    for file in input_files:
        with open(file, "r", encoding="utf-8") as fin:
            for line in fin:
                fout.write(line)

print("Merged into:", output_file)

Merged into: /app/overflow-detection/scripts/data_preprocessing/runs/merged2_7b/samples.jsonl


## Save outputs

In [ ]:

write_jsonl(full_df, FULL_OUTPUT_JSONL)
write_jsonl(filtered_df, FILTERED_OUTPUT_JSONL)

print("Saved full:", FULL_OUTPUT_JSONL)
print("Saved filtered:", FILTERED_OUTPUT_JSONL)


## Optional: download files in Colab

In [ ]:

# from google.colab import files
# files.download(FULL_OUTPUT_JSONL)
# files.download(FILTERED_OUTPUT_JSONL)
